In [ ]:
from fsl.utils.filetree import FileTree
from fsl.wrappers.bet import bet
import subprocess

In [ ]:
import SimpleITK as sitk
from fsl.wrappers import bet
import nibabel as nib
import pydicom
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter,median_filter
from collections import Counter
np.set_printoptions(threshold=np.inf)

In [ ]:
# =========================
# MRI PATH & PET PATH
# =========================
output_dir=r"C:\Users\fhabi\Desktop\PhD\DataSet\nii"
Dataset_root=r"C:\Users\fhabi\Desktop\PhD\DataSet"
input_nii=r"C:\Users\fhabi\Desktop\PhD\DataSet\nii"
output_nii=r"C:\Users\fhabi\Desktop\PhD\DataSet\skull"
mri_root = r"C:\Users\fhabi\Desktop\PhD\DataSet\Dataset\MRI214\ADNI"
pet_root = r"C:\Users\fhabi\Desktop\PhD\DataSet\Dataset\PET214\ADNI"

In [ ]:
def load_nii_files(root):
    for subject in os.listdir(root):

        subject_path = os.path.join(root, subject)

        if not os.path.isdir(subject_path):
            continue

        for pair in os.listdir(subject_path):

            pair_path = os.path.join(subject_path, pair)

            mri_path = os.path.join(pair_path, "mri.nii.gz")
            pet_path = os.path.join(pair_path, "pet.nii.gz")

            # ✔ check files exist
            if not os.path.exists(mri_path):
                print("Missing MRI:", mri_path)
                continue

            if not os.path.exists(pet_path):
                print("Missing PET:", pet_path)
                continue

            # ✔ read images
            mri = sitk.ReadImage(mri_path)
            pet = sitk.ReadImage(pet_path)
           # show_three_views(mri)
           # mri= register_mri_to_pet(mri, pet)
           # show_three_views(mri)
           # show_three_views(pet)
            print("Loaded:", subject, pair)

In [ ]:
def scan_all_file_types(root_folder):
    extensions = {}

    for root, _, files in os.walk(root_folder):
        for f in files:
            ext = os.path.splitext(f)[1].lower()

            if ext == "":
                ext = "NO_EXTENSION"

            extensions[ext] = extensions.get(ext, 0) + 1

    print("File types:")
    for k, v in sorted(extensions.items(), key=lambda x: -x[1]):
        print(k, ":", v)

    return extensions
#scan_all_file_types(pet_root)

In [ ]:
def count_medical_folders(root_folder):
    dicom_folders = set()
    v_folders = set()
    i_folders = set()
    for root, _, files in os.walk(root_folder):

        has_dicom = False
        has_v = False
        has_i = False
        for f in files:
            f_lower = f.lower()

            # DICOM (file-based or extension-based)
            if f_lower.endswith(".dcm") or f_lower.endswith(".dicom"):
                has_dicom = True

            # V files
            if f_lower.endswith(".v"):
                has_v = True
            if f_lower.endswith(".i"):
                has_i = True
        #One time for each folder
        if has_dicom:
            dicom_folders.add(root)

        if has_v:
            v_folders.add(root)

        if has_i:
            i_folders.add(root)

    print("Results:")
    print("DICOM folders:", len(dicom_folders))
    print("V folders:", len(v_folders))
    print("i folders:", len(i_folders))
    print("Total medical folders:", len(dicom_folders) + len(v_folders)+ len(i_folders))

    return dicom_folders, v_folders
#a=count_medical_folders(mri_root)
#b=count_medical_folders(pet_root)

In [ ]:
def count_medical_folders(root_folder):
    dicom_folders = set()
    v_folders = set()
    i_folders = set()
    for root, _, files in os.walk(root_folder):

        has_dicom = False
        has_v = False
        has_i = False
        for f in files:
            f_lower = f.lower()

            # DICOM (file-based or extension-based)
            if f_lower.endswith(".dcm") or f_lower.endswith(".dicom"):
                has_dicom = True

            # V files
            if f_lower.endswith(".v"):
                has_v = True
            if f_lower.endswith(".i"):
                has_i = True

        # One time for each Folder
        if has_dicom:
            dicom_folders.add(root)

        if has_v:
            v_folders.add(root)

        if has_i:
            i_folders.add(root)

    print("Results:")
    print("DICOM folders:", len(dicom_folders))
    print("V folders:", len(v_folders))
    print("i folders:", len(i_folders))
    print("Total medical folders:", len(dicom_folders) + len(v_folders)+ len(i_folders))

    return dicom_folders, v_folders
#a=count_medical_folders(mri_root)
#b=count_medical_folders(pet_root)

In [ ]:
def print_unique_v_sizes(root_folder, dtype='>i2'):
    size_counts = {}   
    examples = {}     

    for root, _, files in os.walk(root_folder):
        for f in files:
            if f.lower().endswith(".i"):
                file_path = os.path.join(root, f)

                try:
                    file_size = os.path.getsize(file_path)

                    best_size = None

                    # Try dif Header
                    for header in [0, 1024, 2048, 4096, 8192]:
                        data = np.fromfile(file_path, dtype=dtype, offset=header)
                        size = data.size
                        print(size)

                        if best_size is None:
                            best_size = size

                    #Count
                    if best_size not in size_counts:
                        size_counts[best_size] = 0
                        examples[best_size] = file_path

                    size_counts[best_size] += 1

                except Exception as e:
                    print("❌ Error:", file_path, e)
    print(best_size)
    print("\nUnique sizes with counts:\n")

    for size, count in sorted(size_counts.items()):
        print(f"Size: {size}  →  Count: {count}")
        print(f"Example file: {examples[size]}")
        print("-" * 50)

    print("Total unique sizes:", len(size_counts))
#print_unique_v_sizes(pet_root)

In [ ]:
def Load_Images_size(folder):

    reader = sitk.ImageSeriesReader()

    dicom_names = reader.GetGDCMSeriesFileNames(folder)
    dicom_names = sorted(dicom_names)

    reader.SetFileNames(dicom_names)

    image = reader.Execute()

    # optional but recommended
    image = sitk.DICOMOrient(image, "LPS")

    # convert to numpy (z, y, x)
    arr = sitk.GetArrayFromImage(image)

    # number of slices
    z = arr.shape[0]
    y = arr.shape[1]
    x = arr.shape[2]

    print("Shape (z, y, x):", arr.shape)
    print("Number of slices (z):", z)

    return image, arr
#Load_Images_size(pet_root)

In [ ]:
def resample_to_size(image, new_size):
    original_size = image.GetSize()
    original_spacing = image.GetSpacing()

    new_spacing = [
        original_spacing[0] * (original_size[0] / new_size[0]),
        original_spacing[1] * (original_size[1] / new_size[1]),
        original_spacing[2] * (original_size[2] / new_size[2]),
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetInterpolator(sitk.sitkLinear)

    return resampler.Execute(image)

In [ ]:
def crop_empty_space(image):
    # پیدا کردن ناحیه غیر صفر
    mask = sitk.OtsuThreshold(image)

    # پیدا کردن bounding box
    label_shape_filter = sitk.LabelShapeStatisticsImageFilter()
    label_shape_filter.Execute(mask)

    bbox = label_shape_filter.GetBoundingBox(1)  
    # (x, y, z, size_x, size_y, size_z)

    cropped = sitk.RegionOfInterest(
        image,
        size=bbox[3:],
        index=bbox[:3]
    )

    return cropped

In [ ]:
def crop_by_threshold(image, threshold=0.09):

    arr = sitk.GetArrayFromImage(image)

    mask = arr > threshold

    if not mask.any():
        return image

    z, y, x = mask.nonzero()

    cropped = arr[
        z.min():z.max()+1,
        y.min():y.max()+1,
        x.min():x.max()+1
    ]

    out = sitk.GetImageFromArray(cropped)

    # ✅ فقط spacing را نگه دار، نه CopyInformation کامل
    out.SetSpacing(image.GetSpacing())

    # ❗ origin و direction را فعلاً حذف کن (safe)
    # یا اگر خواستی بعداً دقیق تنظیم می‌کنیم

    return out

In [ ]:
def center_crop(image, size=(90, 128, 128)):
    """
    Crop center of a 3D image to fixed size (z, y, x)
    """
    array = sitk.GetArrayFromImage(image)

    z, y, x = array.shape
    cz, cy, cx = size

    z_start = (z - cz) // 2
    y_start = (y - cy) // 2
    x_start = (x - cx) // 2

    cropped = array[
        z_start:z_start+cz,
        y_start:y_start+cy,
        x_start:x_start+cx
    ]

    cropped_image = sitk.GetImageFromArray(cropped)
    cropped_image.CopyInformation(image)

    return cropped_image

In [ ]:
def type_conversion(image):
    image = sitk.Cast(image, sitk.sitkFloat32)
    return image

In [ ]:
def show_middle_slice(image, title=""):
    print("ITK size:", image.GetSize())

    arr = sitk.GetArrayFromImage(image)
    arr = np.transpose(arr, (1, 2, 0))
    mid_slice = arr[arr.shape[0] // 2]
    
    print(arr.shape)
    plt.imshow(mid_slice, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
def Orientartion(image,type):
    orientation = sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(image.GetDirection())
    print("Detected "+type+" Image Orientartion:", orientation)                                                                

def Origin(image,type):
    origin= image.GetOrigin()
    print("Detected "+type+" Image Origin:", origin)

def Spacing(image,type):
    spacing=image.GetSpacing()
    print("Detected "+type+" Image Spacing:", spacing)

def Direction(image,type):
    direction=image.GetDirection()
    print("Detected "+type+" Image Direction:",direction)
    
def Image_Info(image,type):
    Orientartion(image,type)
    Origin(image,type)
    Spacing(image,type)
    Direction(image,type)
    
def Set_Image_Info(mri,pet):
    mri.SetOrigin(pet.GetOrigin())
    mri.SetSpacing(pet.GetSpacing())
    mri.SetDirection(pet.GetDirection())

In [ ]:
def register_mri_to_pet(mri, pet):
    
    # ---------- initial alignment ----------
    initial_transform = sitk.CenteredTransformInitializer(
        pet,   # fixed (target space)
        mri,   # moving
        sitk.Euler3DTransform(),                           #rotation چرخش and translation انتقال
        sitk.CenteredTransformInitializerFilter.GEOMETRY   #.GEOMETRY مرکز تصویر را بر اساس هندسه (ابعاد و سایز) تنظیم می‌کند  
        # به شدت پیکسل وابسته نیست (برخلاف MOMENTS) MOMENTS
    )
    #print(sitk.CenteredTransformInitializer(pet, mri, sitk.Euler3DTransform()))

    # ---------- registration method ----------
    registration = sitk.ImageRegistrationMethod()

    ## similarity metric (خوب برای multi-modality)
    registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)

    # optimizer      
    registration.SetOptimizerAsRegularStepGradientDescent(
        learningRate=0.5,
        minStep=1e-4,
        numberOfIterations=100,
        gradientMagnitudeTolerance=1e-6
    )
    registration.SetOptimizerScalesFromPhysicalShift()

    # interpolation
    registration.SetInterpolator(sitk.sitkLinear)

    # initial transform
    registration.SetInitialTransform(initial_transform, inPlace=False)

    # multiresolution     
    registration.SetShrinkFactorsPerLevel([4, 2, 1])
    registration.SetSmoothingSigmasPerLevel([2, 1, 0])
    registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    # ---------- run registration ----------
    final_transform = registration.Execute(pet, mri)
    # fixed=pet, moving=mri
    identity = sitk.Transform(3, sitk.sitkIdentity)
    # ---------- resample MRI into PET space ----------
    reference = sitk.Resample(
    pet,
    mri,
    sitk.Transform(),
    sitk.sitkLinear
)
   # mri_registered = sitk.Resample(
   #     mri,            # moving
   #     pet,            # reference (target space)
   #     sitk.Transform(),
   #     sitk.sitkLinear,
   #     0.0,
   #     sitk.sitkFloat32
   # )
    mri_registered = sitk.Resample(
    mri,
    pet,        # 👈 نه مستقیم pet اگر size مشکل دارد
    final_transform,
    sitk.sitkLinear,
    0.0,
    mri.GetPixelID()
)
    #Image_Info(mri,"MRI2")
    #Image_Info(pet,"PET2")
    #Image_Info(mri_registered,"mri_registered")
    return mri_registered

In [ ]:
def filter(image):
    image = sitk.Median(image, [3, 3, 3]) 
    image = sitk.DiscreteGaussian(image, variance=1.0)
    return image

In [ ]:
def show_slices_step(mri, pet, step=10):
    mri = convert_to_NumPy(mri)
    pet = convert_to_NumPy(pet)

    print("MRI shape:", mri.shape)
    print("PET shape:", pet.shape)

    # تعداد مینیمم برای هماهنگی
    min_slices = min(mri.shape[0], pet.shape[0])

    for i in range(0, min_slices, step):
        
        mri_s = mri[i]
        pet_s = pet[i]

        # نرمال‌سازی
        mri_s = (mri_s - mri_s.min()) / (mri_s.max() - mri_s.min() + 1e-8)
        pet_s = (pet_s - pet_s.min()) / (pet_s.max() - pet_s.min() + 1e-8)

        plt.figure(figsize=(6,3))

        plt.subplot(1,2,1)
        plt.imshow(mri_s, cmap="gray")
        plt.title(f"MRI slice {i}")
        plt.axis("off")

        plt.subplot(1,2,2)
        plt.imshow(pet_s, cmap="gray")
        plt.title(f"PET slice {i}")
        plt.axis("off")

        plt.show()

In [ ]:
def resample_to_pet(mri, pet):

    resampled = sitk.Resample(
        mri,                      # moving
        pet,                      # reference (target grid)
        sitk.Transform(),         # identity (فقط تغییر grid)
        sitk.sitkLinear,
        0.0,
        sitk.sitkFloat32
    )

    return resampled

In [ ]:
def simple_brain_crop(mri):
    import SimpleITK as sitk

    # smoothing
    mri_smooth = sitk.CurvatureAnisotropicDiffusion(mri)

    # 🔥 درست: Otsu threshold به صورت filter
    otsu = sitk.OtsuThresholdImageFilter()
    otsu.SetInsideValue(0)
    otsu.SetOutsideValue(1)

    binary = otsu.Execute(mri_smooth)

    # connected components
    cc = sitk.ConnectedComponent(binary)
    relabel = sitk.RelabelComponent(cc)
    largest = relabel == 1

    # crop bounding box
    label_shape = sitk.LabelShapeStatisticsImageFilter()
    label_shape.Execute(largest)

    bbox = label_shape.GetBoundingBox(1)

    cropped = sitk.RegionOfInterest(
        mri,
        size=bbox[3:],
        index=bbox[0:3]
    )

    return cropped

In [ ]:
def rough_brain_crop(img):
    import SimpleITK as sitk

    mask = sitk.BinaryThreshold(img, lowerThreshold=50)

    cc = sitk.ConnectedComponent(mask)
    relabel = sitk.RelabelComponent(cc)
    largest = relabel == 1

    return sitk.Mask(img, largest)

In [ ]:
def Count_Size():
    shape_counter = Counter()
    count = 0

    subjects = os.listdir(pet_root)
    print("Total subjects:", len(subjects))

    for i, subject in enumerate(subjects):
        mri_path = os.path.join(mri_root, subject)
        pet_path = os.path.join(pet_root, subject)

        if not os.path.isdir(mri_path):
            continue
        if not os.path.exists(pet_path):
            continue

        mri_folders = []

        # 🔍 collect ALL MRI DICOM folders
        for d, _, _ in os.walk(pet_path):
            if is_dicom_folder(d):
                mri_folders.append(d)

        # 🧠 process ALL MRI scans for this subject
        for mri_folder in mri_folders:
            mri = Load_Images(mri_folder)
            mri = convert_to_NumPy(mri)

            shape_counter[mri.shape] += 1
            count += 1

            #print(f"Subject {i} MRI shape:", mri.shape)


    # ✅ Final results
    print("\nTotal MRI series counted:", count)

    print("\n=== Shape Counts ===")
    for shape, c in shape_counter.most_common():
        print(f"{shape} : {c}")
#Count_Size()

In [ ]:
def find_best_header(file_path, slices=63, rows=128, cols=128, dtype='>i2'):
    total_bytes = os.path.getsize(file_path)

    voxel_size = np.dtype(dtype).itemsize
    base = slices * rows * cols * voxel_size

    best_header = None
    best_remainder = 1e18

    for header in range(0, 8192, 2):  # search small range
        data_bytes = total_bytes - header

        if data_bytes <= 0:
            continue

        if data_bytes % base == 0:
            print("✅ Perfect header found:", header)
            return header

        remainder = data_bytes % base

        if remainder < best_remainder:
            best_remainder = remainder
            best_header = header

    print("⚠️ Best approximate header:", best_header)
    return best_header

In [ ]:
def resample_to_reference(fixed, moving):

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed)
    resampler.SetInterpolator(sitk.sitkLinear)

    return resampler.Execute(moving)

In [ ]:
#--------------------------------------------------------------------

In [ ]:
def detect_format(folder):
    files = os.listdir(folder)
    
    if any(f.lower().endswith('.dcm') for f in files):
        return ".dicom"
    elif any(f.lower().endswith('.v') for f in files):
        return ".v"
    elif any(f.lower().endswith('.i') for f in files):
        return ".i"
    elif any(f.lower().endswith('.hdr') for f in files):
        return ".hdr"
    else:
        return None

In [ ]:
def is_dynamic_pet(folder):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)

    reader.SetFileNames(dicom_names)

    # گرفتن متادیتا از اولین فایل
    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()

    reader.Execute()

    # tag مهم برای تعداد فریم‌ها
    try:
        num_frames = int(reader.GetMetaData(0, "0054|0101"))  # Number of Time Slices
        return num_frames > 1
    except:
        return False

In [ ]:
def has_time_dimension(folder):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)

    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    reader.SetFileNames(dicom_names)

    reader.Execute()

    times = []

    for i in range(len(dicom_names)):
        if reader.HasMetaDataKey(i, "0054|1300"):  # Frame Reference Time
            times.append(reader.GetMetaData(i, "0054|1300"))

    return len(set(times)) > 1

In [ ]:
def detect_pet_type(folder):
    if is_dynamic_pet(folder):
        return "Dynamic PET"
    elif has_time_dimension(folder):
        return "Dynamic PET"
    else:
        return "Static PET"

In [ ]:
def force_spacing_222(image):

    original_size = image.GetSize()
    original_spacing = image.GetSpacing()

    new_spacing = (2.0, 2.0, 2.0)

    new_size = [
        int(original_size[i] * (original_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]

    resampled = sitk.Resample(
        image,
        new_size,
        sitk.Transform(),
        sitk.sitkLinear,
        image.GetOrigin(),
        new_spacing,
        image.GetDirection(),
        0,
        image.GetPixelID()
    )

    return resampled

In [ ]:
def load_dicom_file(folder,mode):
    frames=1
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)
    n = len(dicom_names)
    if n == 0:
        return None

    if mode=="pet" and is_dynamic_pet(folder):
        dicom_names = dicom_names[:n // 6]
        frames=6
    reader.SetFileNames(dicom_names)
    try:
        image = reader.Execute()
    except:
        return None
    
    #image=crop_empty_space(image)
    image = sitk.DICOMOrient(image, "RAI")           #Orientation : RAS   #LPS
    if mode=="pet" and (image.GetSize()==(128, 128, 47) or image.GetSize()==(400, 400, 109) or image.GetSize()==(128, 128, 31)):
        image = sitk.DICOMOrient(image, "RAS") 
    image = sitk.Cast(image, sitk.sitkFloat32)       #Type conversion
    image = sitk.RescaleIntensity(image, 0, 1)

    #image = sitk.Median(image, [1, 1, 1])            #Filter denoising
    image = sitk.DiscreteGaussian(image, variance=1.0)
    original = image
    arr = sitk.GetArrayFromImage(image)              #Normalization
    arr = arr.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) 
    arr = np.where(arr < 0.01, 0, arr)
    image = sitk.GetImageFromArray(arr)
    image.CopyInformation(original)

    return (image,frames)

In [ ]:
def load_v_file(folder,  mode="",frames=6, slices=63, rows=128, cols=128, header=0):
    v_files = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".v"):
                v_files.append(os.path.join(root, f))

    if len(v_files) == 0:
        raise FileNotFoundError("No .v file found")

    # read data
    file_path = v_files[0]
    data = np.fromfile(file_path, dtype='>i2', offset=header)
    data = data.astype(np.float32)
 
    #print("Raw Data size:", data.size)
    

    base = slices * rows * cols
    frames = data.size // base
    usable_size = frames * base
    #if (data.size-usable_size!=2048):
    #    print( data.size-usable_size)
    #    print("Raw Data size:", data.size)
    #    print("Detected frames:", frames)

    if frames == 0:
        raise ValueError("❌ Cannot reshape data")

    #trim extra data
    #data = data[-usable_size:]
    data = data[-usable_size:]
    #print("Original size:", data.size)
    #print("Usable size:", usable_size)
    #print("Removed extra:", data.size - usable_size)

    # reshape safely
    vol = data.reshape((frames, slices, rows, cols))

    # take first frame
    arr = vol[-1]
    arr = np.flip(arr, axis=1)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) 
    arr = np.where(arr < 0.01, 0, arr)
    
    # convert to SimpleITK
    image = sitk.GetImageFromArray(arr)
    image = sitk.DICOMOrient(image, "RPS")
    image.SetSpacing((2.1, 2.1, 2.4))
    image.SetOrigin((128.0, 128.0, 75.6))
    image.SetDirection([-1,0,0,
                  0,-1,0,
                  0,0,-1])
    return (image,frames)

In [ ]:
def load_i_file(folder, mode="",dtype=np.float32, slices=207, rows=256,cols=256,header=0):
    i_files = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".i"):
                i_files.append(os.path.join(root, f))

    if len(i_files) == 0:
        raise FileNotFoundError("No .i file found")
    
    # read data
    file_path = i_files[0]
    data = np.fromfile(file_path, dtype=dtype, offset=header)
    data = data.astype(np.float32)
    #print("Raw Data size:", data.size)

    base = slices * rows * cols

    frames = data.size // base
    usable_size = frames * base

    #print("Detected frames:", frames)

    if frames == 0:
        raise ValueError("❌ Cannot reshape data")

    data = data[-usable_size:]

    # reshape safely
    vol = data.reshape((frames, slices, rows, cols))

    # take first frame
    arr = vol[-1]
    arr = np.flip(arr, axis=1)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) 
    arr = np.where(arr < 0.01, 0, arr)  #all pix less than 0.01 set 0
    # convert to SimpleITK
    image = sitk.GetImageFromArray(arr)
    image = sitk.DICOMOrient(image, "RPS")
    return (image,frames)

In [ ]:
def load_analyze_as_sitk(hdr_path):
    img = sitk.ReadImage(hdr_path)
    img = sitk.DICOMOrient(img, "LPS")  # optional fix
    return img

In [ ]:
def Load_Images(folder, fmt, mode):

    if fmt == ".dicom":
        return load_dicom_file(folder, mode)
    elif fmt == ".v":
        return load_v_file(folder, mode)
    elif fmt == ".i":
        return load_i_file(folder, mode)
    else:
        raise ValueError("Unknown format")

In [ ]:
def convert_to_NumPy(img):
    if img is None:
        raise ValueError("Image is None")

    if isinstance(img, np.ndarray):
        return img

    return sitk.GetArrayFromImage(img)

In [ ]:
def show_three_views(vol):
    vol = convert_to_NumPy(vol)
    z, y, x = vol.shape
    print(vol.shape)
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(vol[z//2], cmap='gray')
    plt.title("Axial")

    plt.subplot(1,3,2)
    plt.imshow(vol[:, y//2, :], cmap='gray')
    plt.title("Coronal")

    plt.subplot(1,3,3)
    plt.imshow(vol[:, :, x//2], cmap='gray')
    plt.title("Sagittal")

    plt.show()

In [ ]:
def get_slice(volume):
    
    mid = volume.shape[0] // 2
    sl = volume[mid]

    return sl, mid   

In [ ]:
def plot_pair(mri, pet, title=""):

    # FORCE conversion ONLY ONCE
    mri=convert_to_NumPy(mri)
    pet=convert_to_NumPy(pet)

    print("MRI shape:", mri.shape)
    print("PET shape:", pet.shape)

    mri_s, mri_idx = get_slice(mri)
    pet_s, pet_idx = get_slice(pet)
    
  

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(mri_s, cmap="gray" )
    plt.title(f"MRI (slice {mri_idx})")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(pet_s, cmap="gray" )
    plt.title(f"PET (slice {pet_idx})")
    plt.axis("off")

    plt.suptitle(title)
    plt.show()

In [ ]:
def Create_dataset_nii(mri_np,pet_np,subject,count):
    affine = np.eye(4)

    mri_nii = nib.Nifti1Image(mri_np, affine)
    pet_nii = nib.Nifti1Image(pet_np, affine)

    subject_out = os.path.join(output_dir, subject)
    pair_out = os.path.join(subject_out, f"pair_{count:02d}")

    mri_out_dir = os.path.join(pair_out, "mri")
    pet_out_dir = os.path.join(pair_out, "pet")

    os.makedirs(mri_out_dir, exist_ok=True)
    os.makedirs(pet_out_dir, exist_ok=True)

    mri_path = os.path.join(mri_out_dir, "mri.nii.gz")
    pet_path = os.path.join(pet_out_dir, "pet.nii.gz")

    nib.save(mri_nii, mri_path)
    nib.save(pet_nii, pet_path)

In [ ]:
def skull_strip_remove_neck(input_img, output_img, frac=0.3):
    cmd = [
        "bet",
        input_img,
        output_img,
        "-R",   # robust center estimation
        "-B",   # bias field + better cleanup
        "-f", str(frac),  # fractional intensity (try 0.25–0.4)
        "-g", "0",         # vertical gradient (important for neck removal)
        "-m"   # also output mask
    ]
    subprocess.run(cmd, check=True)

In [ ]:
def remove_zero_padding(image,pix,img_type):
    """
    Crop 3D image by removing all-zero background using bounding box.
    """
    
    img_np = sitk.GetArrayFromImage(image)
    z = img_np.shape[0]
    #img_norm = (img_np - img_np.mean()) / (img_np.std() + 1e-8)
   
    if img_type == "mri":
        img_np = img_np[:4*z//5, :, :]
    else:
        img_np = img_np[:z, :, :]
    img_norm = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    # find all non-zero voxels
    coords = np.where(img_norm >= pix)

    if len(coords[0]) == 0:
        raise ValueError("Image is completely zero")

    zmin, zmax = coords[0].min(), coords[0].max()
    ymin, ymax = coords[1].min(), coords[1].max()
    xmin, xmax = coords[2].min(), coords[2].max()
    
    # crop volume
    cropped = img_norm[
        zmin:zmax+1,
        ymin:ymax+1,
        xmin:xmax+1
    ]
    cropped = (cropped - cropped.min()) / (cropped.max() - cropped.min() + 1e-8)
    # convert back to SimpleITK
    out = sitk.GetImageFromArray(cropped)

    # keep spatial info
    out.SetSpacing(image.GetSpacing())
    out.SetDirection(image.GetDirection())

    return out

In [ ]:
def Load_png(path):

    img = plt.imread(path)
    return img

In [ ]:
def downsample_image(img, new_size):
    print(new_size)
    img = sitk.Cast(img, sitk.sitkFloat32)

    old_size = img.GetSize()
    old_spacing = img.GetSpacing()

    new_spacing = [
        old_spacing[i] * (old_size[i] / new_size[i])
        for i in range(3)
    ]

    resampled= sitk.Resample(
        img,
        new_size,
        sitk.Transform(),
        sitk.sitkLinear,
        img.GetOrigin(),
        new_spacing,
        img.GetDirection(),
        0,
        sitk.sitkFloat32
    )

    return resampled

In [ ]:
def Creat_nii_Dataset(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    count = 0

    subjects = os.listdir(mri_root)
    #print("Total subjects:", len(subjects))

    for i, subject in enumerate(subjects):
        print("Subject",i+1,":", subject)
        pair_count=1
        mri_path = os.path.join(mri_root, subject)
        pet_path = os.path.join(pet_root, subject)

        #subject_out = os.path.join(output_dir, subject)
        #os.makedirs(subject_out, exist_ok=True)

        if not os.path.isdir(mri_path):
            continue
        if not os.path.exists(pet_path):
            continue

        if os.path.isdir(mri_path) and os.path.exists(pet_path):

            mri_folders = []
            pet_folders = []

            # collect all MRI folders
            for d, _, _ in os.walk(mri_path):
                fmt = detect_format(d)
                if fmt:
                    mri_folders.append(d)

            # collect all PET folders
            for d, _, _ in os.walk(pet_path):
                fmt = detect_format(d)
                if fmt:
                    pet_folders.append(d)

            # loop over pairs
            for idx,(mri_folder, pet_folder) in enumerate(zip(mri_folders, pet_folders)):
            

                mri_fmt = detect_format(mri_folder)
                pet_fmt = detect_format(pet_folder)
                print("Detected PET Image format:", pet_fmt)
                
                
                if pet_fmt==".dicom":
                    mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
                    pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
                    subject_out = os.path.join(output_dir, subject)
                    os.makedirs(subject_out, exist_ok=True)
                    pair_out = os.path.join(subject_out, f"pair_{idx+1}")
                    os.makedirs(pair_out, exist_ok=True)

                    # ✔ save correctly
                    sitk.WriteImage(mri, os.path.join(pair_out, "mri.nii.gz"))
                    sitk.WriteImage(pet, os.path.join(pair_out, "pet.nii.gz"))
                    print(pair_count)
                    pair_count+=1
            count += 1
    print ("Total Pair Images:",count) 
#Creat_nii_Dataset(output_dir)

In [ ]:
def Load_NIfTI_Image_File(input_nii):
     image = sitk.ReadImage(input_nii)
     return image

In [ ]:
def v(input_nii):
    for subject in os.listdir(input_nii):

        subject_path = os.path.join(input_nii, subject)

        if not os.path.isdir(subject_path):
            continue

        for pair in os.listdir(subject_path):

            pair_path = os.path.join(subject_path, pair)

            mri_path = os.path.join(pair_path, "mri.nii.gz")
            pet_path = os.path.join(pair_path, "pet.nii.gz")

            # ✔ check files exist
            if not os.path.exists(mri_path):
                print("Missing MRI:", mri_path)
                continue

            if not os.path.exists(pet_path):
                print("Missing PET:", pet_path)
                continue

            # ✔ read images
            mri = Load_NIfTI_Image_File(mri_path)
            pet = Load_NIfTI_Image_File(pet_path)
            
            print("Loaded:", subject, pair)
            mri= register_mri_to_pet(mri, pet)
            show_three_views(mri)
            show_three_views(pet)
        

In [ ]:
def rule(values):
    # if max in window is 1 → set all to 1
    if np.max(values) == 1:
        return 1
    else:
        return 0

In [ ]:
from scipy.ndimage import convolve
def Hist(Image):

    plt.hist(Image, bins=100)

    plt.title("PET Histogram")
    plt.xlabel("Intensity")
    plt.ylabel("Voxel Count")
    plt.show()
    counts, bins = np.histogram(Image, bins=100)
    max_bin_idx = np.argmax(counts)
    left_idx = max(0, max_bin_idx - 4)
    right_idx = min(len(bins)-1, max_bin_idx + 5)
    bin_min = bins[left_idx]
    bin_max = bins[right_idx]
    mask = (Image >= bin_min) & (Image < bin_max)
    Image[mask] = 1
    arr = sitk.GetArrayFromImage(Image)
    arr = Image.copy()

  

    kernel = np.ones((2,2,2))

    count = convolve(arr.astype(np.int32), kernel, mode='constant')

    filtered = (count > 4).astype(np.uint8)
    return Image

In [ ]:
count = 0
subjects = os.listdir(mri_root)
#print("Total subjects:", len(subjects))

for i, subject in enumerate(subjects):
    #print("Subject",i+1,":", subject)
    pair_count=1
    mri_path = os.path.join(mri_root, subject)
    pet_path = os.path.join(pet_root, subject)
   
    if not os.path.isdir(mri_path):
        continue
    if not os.path.exists(pet_path):
        continue

    if os.path.isdir(mri_path) and os.path.exists(pet_path):

        mri_folders = []
        pet_folders = []

        # collect all MRI folders
        for d, _, _ in os.walk(mri_path):
            fmt = detect_format(d)
            if fmt:
                mri_folders.append(d)

        # collect all PET folders
        for d, _, _ in os.walk(pet_path):
            fmt = detect_format(d)
            if fmt:
                pet_folders.append(d)

        #print(f"Found {len(mri_folders)} MRI and {len(pet_folders)} PET")

        # loop over pairs
        for mri_folder, pet_folder in zip(mri_folders, pet_folders):

            #print("Detected MRI Image folder:", mri_folder)
            #print("Detected PET Image folder:", pet_folder)
            
            mri_fmt = detect_format(mri_folder)
            #print("Detected MRI Image format:", mri_fmt)
            pet_fmt = detect_format(pet_folder)
            #print("Detected PET Image format:", pet_fmt)
            #mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
            #pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
            if pet_fmt==".i" :
                print("Subject",i+1,":", subject)
                mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
                pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
                #print(subject)
            #if(pet_frames!=6):
                show_three_views(mri)
                mri= register_mri_to_pet(mri, pet)
                show_three_views(mri)
                show_three_views(pet)
           # Image_Info(mri,"MRI1")
           # Image_Info(pet,"PET1")
           # mri, pet, mask=skull_strip_mri_pet(mri, pet)

            #print("Detected MRI Image Frames:", mri_frames)
            #print("Detected PET Image Frames:", pet_frames)

            #mri_np = convert_to_NumPy(mri)
            #pet_np = convert_to_NumPy(pet)
           
            #print("Detected MRI Image shape before crop:", mri_np.shape)
            #print("Detected PET Image shape before crop:", pet_np.shape)

            #print("MRI min:", mri_np.min())
            #print("MRI max:", mri_np.max())
            #print("PET min:", pet_np.min())
            #print("PET max:", pet_np.max())
            ##mri= remove_zero_padding(mri,0.3,"mri")
            #pet= remove_zero_padding(pet,0.2,"pet")
            #
            #mri_crop_np = convert_to_NumPy(mri)
            #pet_crop_np = convert_to_NumPy(pet)
#
            #print("Detected MRI Image shape after crop:", mri_crop_np.shape)
            #print("Detected PET Image shape after crop:", pet_crop_np.shape)
#
            ##Create_dataset_nii(mri_np,pet_np,subject,pair_count)
            #print("MRI min:", mri_crop_np.min())
            #print("MRI max:", mri_crop_np.max())
#
            #print("PET min:", pet_crop_np.min())
            #print("PET max:", pet_crop_np.max())
            
            #show_three_views(mri)
            #show_three_views(pet)
            #print(mri_crop_np.dtype)
            #print(pet_crop_np.dtype)
            #plot_pair(mri, pet, title=subject)
            #mri = sitk.DICOMOrient(mri, "RAS")
            #pet = sitk.DICOMOrient(pet, "RAS")
   
           # 
           # x,y,z=pet_crop_np.shape
            #mri=downsample_image(mri,(z,y,x))
            #Set_Image_Info(mri,pet)
            #Image_Info(pet,"PET")
            #plot_pair(mri, pet, title=subject)
            #Set_Image_Info(mri,pet)
            #pet.SetDirection(mri.GetDirection())
            #pet.SetOrigin((0,0,0))
            #plot_pair(mri, pet, title=subject)
            #show_three_views(pet)
            #show_three_views(mri)
            #mri= register_mri_to_pet(mri, pet)
            #mri=force_spacing_111(mri)
            #show_three_views(mri)
            #show_three_views(pet)
            #print(mri_crop_np.dtype)
            #print(pet_crop_np.dtype)
            #plot_pair(mri, pet, title=subject)

            
            pair_count+=1
            count += 1
            print(count)
        #if count==2:
                #break

print ("Total Pair Images:",count)   